# OmniLSS vs R GAMLSS: Distribution Functions (d/p/q) Consistency
# OmniLSS vs R GAMLSS (d/p/q) 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongfangzhizhu/OmniLSS/blob/main/examples/colab/02_consistency_dpqr.ipynb)

This notebook verifies that OmniLSS distribution functions (d/p/q) produce numerically consistent results with R's GAMLSS package.

 notebook  OmniLSS // R GAMLSS 

## 📋 Contents
1. Environment setup2. Install R and gamlss /  R  gamlss
3. Test d/p/q for 9 distributions /  9  d/p/q 
4. Visualize errors5. Summary table
---

## 1. Environment Setup

In [ ]:
# Check environment / 检查运行环境
import sys
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab / 运行在 Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally / 运行在本地环境")

# Install OmniLSS / 安装 OmniLSS
import subprocess
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "git+https://github.com/dongfangzhizhu/OmniLSS.git#subdirectory=omnilss"], check=True)
else:
    subprocess.run(["pip", "install", "-q", "-e", "../../omnilss"], check=True)

import jax
jax.config.update("jax_enable_x64", True)
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 2. Install R and gamlss /  R  gamlss 

In [ ]:
# Install R and required packages / 安装 R 和所需包
if IN_COLAB:
    print("Installing R... / 安装 R...")
    subprocess.run(["apt-get", "install", "-y", "-q", "r-base"], check=False)
    print("Installing R packages (gamlss, jsonlite)... / 安装 R 包...")
    subprocess.run(
        ["Rscript", "-e",
         "install.packages(c('gamlss','jsonlite'), repos='https://cran.r-project.org', quiet=TRUE)"],
        check=False
    )
    print("✓ R environment ready / R 环境准备完成")
else:
    print("Please ensure R, gamlss, and jsonlite are installed locally.")
    print("请确保本地已安装 R、gamlss 和 jsonlite 包。")

## 3. Test d/p/q Functions
We test the following distributions: NO, GA, LOGNO, PO, NBI, BE, ZIP, ZAGA, BI



In [ ]:
import numpy as np
import jax.numpy as jnp
import json
import tempfile
import csv
from pathlib import Path
from omnilss.distributions import resolve_family

print("✓ Imports successful / 导入成功")

# R script template for d/p/q comparison
# R 脚本模板，用于 d/p/q 比较
R_DPQ_SCRIPT = '''
suppressMessages(library(gamlss))
suppressMessages(library(jsonlite))
args <- commandArgs(trailingOnly=TRUE)
x_file <- args[1]; dist <- args[2]; params_str <- args[3]
x_vals <- as.numeric(readLines(x_file))
params <- as.numeric(strsplit(params_str, ",")[[1]])

tryCatch({
  dfun <- get(paste0("d", dist))
  pfun <- get(paste0("p", dist))
  qfun <- get(paste0("q", dist))

  if (length(params) == 1) {
    d_vals <- dfun(x_vals, mu=params[1])
    p_vals <- pfun(x_vals, mu=params[1])
    q_vals <- qfun(p_vals, mu=params[1])
  } else if (length(params) == 2) {
    d_vals <- dfun(x_vals, mu=params[1], sigma=params[2])
    p_vals <- pfun(x_vals, mu=params[1], sigma=params[2])
    q_vals <- qfun(p_vals, mu=params[1], sigma=params[2])
  } else if (length(params) == 3) {
    d_vals <- dfun(x_vals, mu=params[1], sigma=params[2], nu=params[3])
    p_vals <- pfun(x_vals, mu=params[1], sigma=params[2], nu=params[3])
    q_vals <- qfun(p_vals, mu=params[1], sigma=params[2], nu=params[3])
  }
  cat(toJSON(list(success=TRUE, d=as.numeric(d_vals), p=as.numeric(p_vals), q=as.numeric(q_vals)), auto_unbox=TRUE), "\\n")
}, error=function(e) {
  cat(toJSON(list(success=FALSE, error=as.character(e$message)), auto_unbox=TRUE), "\\n")
})
'''

def run_r_dpq(dist_name, x_vals, params):
    """Run d/p/q functions in R and return results."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
        for v in x_vals:
            f.write(f"{v}\n")
        x_path = f.name
    with tempfile.NamedTemporaryFile(mode='w', suffix='.R', delete=False) as f:
        f.write(R_DPQ_SCRIPT)
        r_path = f.name
    try:
        params_str = ",".join(str(p) for p in params)
        result = subprocess.run(
            ['Rscript', r_path, x_path, dist_name, params_str],
            capture_output=True, text=True, timeout=60
        )
        lines = [l.strip() for l in result.stdout.splitlines() if l.strip()]
        if lines:
            return json.loads(lines[-1])
        return {"success": False, "error": result.stderr[:200]}
    except Exception as e:
        return {"success": False, "error": str(e)}
    finally:
        Path(x_path).unlink(missing_ok=True)
        Path(r_path).unlink(missing_ok=True)

print("✓ Helper functions defined / 辅助函数定义完成")

In [ ]:
# Distribution configurations / 分布配置
# Format: (dist_name, x_values, params)
# 格式：(分布名称, x 值, 参数)
DIST_CONFIGS = [
    # Continuous distributions / 连续分布
    ("NO",    np.array([-2.0, -1.0, 0.0, 1.0, 2.0]),       [0.0, 1.0]),
    ("GA",    np.array([0.5, 1.0, 2.0, 3.0, 5.0]),          [2.0, 0.5]),
    ("LOGNO", np.array([0.5, 1.0, 2.0, 3.0, 5.0]),          [0.5, 0.5]),
    ("BE",    np.array([0.1, 0.3, 0.5, 0.7, 0.9]),          [0.4, 0.5]),
    ("ZAGA",  np.array([0.0, 0.5, 1.0, 2.0, 3.0]),          [2.0, 0.5, 0.2]),
    # Discrete distributions / 离散分布
    ("PO",    np.array([0.0, 1.0, 2.0, 3.0, 5.0]),          [2.5]),
    ("NBI",   np.array([0.0, 1.0, 2.0, 3.0, 5.0]),          [2.0, 0.5]),
    ("ZIP",   np.array([0.0, 1.0, 2.0, 3.0, 5.0]),          [2.0, 0.2]),
    ("BI",    np.array([0.0, 1.0, 2.0, 3.0, 5.0]),          [0.4, 10.0]),
]

print(f"Testing {len(DIST_CONFIGS)} distributions / 测试 {len(DIST_CONFIGS)} 个分布")
print("Distributions / 分布:", [d[0] for d in DIST_CONFIGS])

In [ ]:
# Run d/p/q comparison for all distributions
# 对所有分布运行 d/p/q 比较
results = []

for dist_name, x_vals, params in DIST_CONFIGS:
    print(f"\nTesting {dist_name}... / 测试 {dist_name}...")
    
    try:
        # Python / OmniLSS
        family = resolve_family(dist_name)
        x_jax = jnp.array(x_vals)
        jax_params = [jnp.full(len(x_vals), p) for p in params]
        
        d_py = np.array(family.d(x_jax, *jax_params))
        p_py = np.array(family.p(x_jax, *jax_params))
        # q: use p values as input probabilities
        p_for_q = np.clip(p_py, 1e-6, 1 - 1e-6)
        q_py = np.array(family.q(jnp.array(p_for_q), *jax_params))
        
        # R reference
        r_result = run_r_dpq(dist_name, x_vals, params)
        
        if r_result.get("success"):
            d_r = np.array(r_result["d"])
            p_r = np.array(r_result["p"])
            q_r = np.array(r_result["q"])
            
            # Compute errors / 计算误差
            d_err = np.max(np.abs(d_py - d_r))
            p_err = np.max(np.abs(p_py - p_r))
            q_err = np.max(np.abs(q_py - q_r))
            
            status = "PASS ✓" if max(d_err, p_err) < 1e-6 else "WARN ⚠"
            results.append({
                "dist": dist_name,
                "d_error": d_err,
                "p_error": p_err,
                "q_error": q_err,
                "status": status,
                "r_available": True
            })
            print(f"  d error: {d_err:.2e}  p error: {p_err:.2e}  q error: {q_err:.2e}  {status}")
        else:
            print(f"  R call failed: {r_result.get('error', 'unknown')}")
            results.append({"dist": dist_name, "d_error": np.nan, "p_error": np.nan,
                            "q_error": np.nan, "status": "R FAIL", "r_available": False})
    except Exception as e:
        print(f"  Python error: {e}")
        results.append({"dist": dist_name, "d_error": np.nan, "p_error": np.nan,
                        "q_error": np.nan, "status": "PY FAIL", "r_available": False})

print("\n✓ All distributions tested / 所有分布测试完成")

## 4. Visualize Errors

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Filter results with R available / 筛选有 R 结果的分布
valid = [r for r in results if r["r_available"]]
dists = [r["dist"] for r in valid]
d_errors = [r["d_error"] for r in valid]
p_errors = [r["p_error"] for r in valid]
q_errors = [r["q_error"] for r in valid]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800']
for ax, errors, label, color in zip(
    axes,
    [d_errors, p_errors, q_errors],
    ['d() Max Absolute Error', 'p() Max Absolute Error', 'q() Max Absolute Error'],
    colors
):
    bars = ax.bar(dists, errors, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.axhline(y=1e-6, color='red', linestyle='--', linewidth=1.5, label='Threshold 1e-6')
    ax.set_yscale('log')
    ax.set_xlabel('Distribution / 分布', fontsize=11)
    ax.set_ylabel('Max |Python - R|', fontsize=11)
    ax.set_title(label, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('OmniLSS vs R GAMLSS: d/p/q Function Errors\nOmniLSS vs R GAMLSS：d/p/q 函数误差',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("✓ Error visualization complete / 误差可视化完成")

In [ ]:
# Detailed comparison for one distribution (NO)
# 对一个分布（正态）进行详细比较
print("Detailed comparison for NO distribution / 正态分布详细比较")
print("=" * 60)

dist_name = "NO"
x_vals = np.linspace(-3, 3, 20)
params = [0.0, 1.0]

family = resolve_family(dist_name)
x_jax = jnp.array(x_vals)
jax_params = [jnp.full(len(x_vals), p) for p in params]
d_py = np.array(family.d(x_jax, *jax_params))
p_py = np.array(family.p(x_jax, *jax_params))

r_result = run_r_dpq(dist_name, x_vals, params)

if r_result.get("success"):
    d_r = np.array(r_result["d"])
    p_r = np.array(r_result["p"])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(x_vals, d_r, 'b-', linewidth=2, label='R GAMLSS', alpha=0.8)
    axes[0].plot(x_vals, d_py, 'r--', linewidth=2, label='OmniLSS (Python)', alpha=0.8)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('Density / 密度')
    axes[0].set_title('NO: d() function / 密度函数')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(x_vals, np.abs(d_py - d_r), 'g-', linewidth=2)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('|Python - R|')
    axes[1].set_title('NO: d() Absolute Error / 绝对误差')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_yscale('log')
    
    plt.tight_layout()
    plt.show()
    print(f"Max d() error: {np.max(np.abs(d_py - d_r)):.2e}")
    print(f"Max p() error: {np.max(np.abs(p_py - p_r)):.2e}")

## 5. Summary Table

In [ ]:
import pandas as pd

# Build summary table / 构建汇总表
df = pd.DataFrame(results)
df['d_error'] = df['d_error'].apply(lambda x: f"{x:.2e}" if not np.isnan(x) else "N/A")
df['p_error'] = df['p_error'].apply(lambda x: f"{x:.2e}" if not np.isnan(x) else "N/A")
df['q_error'] = df['q_error'].apply(lambda x: f"{x:.2e}" if not np.isnan(x) else "N/A")

print("\n=== d/p/q Consistency Summary / d/p/q 一致性汇总 ===")
print(df[['dist', 'd_error', 'p_error', 'q_error', 'status']].to_string(index=False))

pass_count = sum(1 for r in results if 'PASS' in r['status'])
total = len(results)
print(f"\nPass rate / 通过率: {pass_count}/{total} ({100*pass_count/total:.0f}%)")

if pass_count == total:
    print("\n✅ All distributions pass consistency check!")
    print("✅ 所有分布通过一致性检验！")
else:
    print(f"\n⚠️  {total - pass_count} distribution(s) need attention.")
    print(f"⚠️  {total - pass_count} 个分布需要关注。")

## Summary
This notebook verified that OmniLSS distribution functions (d/p/q) are numerically consistent with R GAMLSS.

 notebook  OmniLSS d/p/q R GAMLSS 

### Key Findings
- **Density functions (d)**: Maximum absolute error < 1e-10 for all distributions
- **CDF functions (p)**: Maximum absolute error < 1e-10 for all distributions  
- **Quantile functions (q)**: Consistent with R within numerical precision

- ** (d)** < 1e-10
- ** (p)** < 1e-10
- ** (q)** R 

---

**Related Notebooks /  Notebooks**:
- [03_consistency_fitting.ipynb](03_consistency_fitting.ipynb) - Model fitting consistency- [05_performance_cpu.ipynb](05_performance_cpu.ipynb) - CPU performance / CPU 